# Audit Social Comments NLP v2

Notebook kiểm định dataset bình luận mạng xã hội tiếng Việt theo hướng **NLP có thể kiểm chứng**, thay cho việc coi các rule thủ công là nhãn đúng tuyệt đối.

### Điểm cải thiện chính

- Giữ riêng văn bản gốc, Unicode NFC, khóa dò từ điển không dấu và đầu vào model.
- Dò từ/cụm từ theo ranh giới từ; không còn lỗi từ `cho` bị hiểu thành `chó`.
- Đếm emoji theo grapheme cluster, hỗ trợ emoji ghép và ZWJ.
- Near-duplicate dùng TF-IDF ký tự + nearest neighbors, không lỗi `empty vocabulary`, không tạo full ma trận N×N.
- Template cluster yêu cầu quan hệ gần-clique để hạn chế lỗi chaining.
- Title relevance không dùng độ dài comment làm bằng chứng.
- Toxicity, safety, teencode và slang được xuất dưới dạng **rule signal/candidate review**, không giả làm ground truth.
- Có kiểm thử hồi quy, chia train/validation/test theo group để giảm leakage, fingerprint dữ liệu và log phiên bản.
- CSV xuất ra được trung hòa formula injection.

### Đầu vào mặc định

File `raw.csv` với bốn cột bắt buộc:

```text
row_id, platform, title, comment
```

Kết quả được lưu trong `audit_output_v2/`.


## 1. Cài thư viện

Notebook chạy được với `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `tqdm` và `regex`.
Các thành phần semantic embedding và tách từ nâng cao là tùy chọn, mặc định tắt để notebook chạy ổn định và không tự tải model.


In [ ]:
# Không cài package tự động khi Run All để notebook có thể chạy offline.
# Nếu môi trường thiếu package bắt buộc, cài một lần bằng lệnh sau rồi restart kernel:
# %pip install pandas numpy scikit-learn matplotlib tqdm regex
# Chỉ cần khi bật CFG.enable_semantic_embeddings/CFG.enable_word_segmentation:
# %pip install sentence-transformers underthesea


## 2. Import và cấu hình


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import logging
import math
import os
import platform as py_platform
import random
import re
import sys
import unicodedata
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm

try:
    import regex as uregex
except ImportError as exc:
    raise ImportError(
        "Thiếu package 'regex'. Cài bằng: pip install regex"
    ) from exc

warnings.filterwarnings('default')

@dataclass(frozen=True)
class AuditConfig:
    input_file: str = ''  # tự dò raw.csv; có thể ghi đè bằng đường dẫn tuyệt đối
    output_dir: str = 'audit_output_v2'
    gold_labels_file: str = 'gold_labels.csv'
    random_seed: int = 42
    n_jobs: int = 1  # an toàn khi chạy local/Windows; tránh worker process ngầm

    # NLP tùy chọn
    enable_word_segmentation: bool = False
    enable_semantic_embeddings: bool = False
    embedding_model_name: str = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

    # Near duplicate
    near_dup_char_threshold: float = 0.86
    near_dup_combined_threshold: float = 0.84
    near_dup_neighbors: int = 15
    near_dup_min_chars: int = 8

    # N-gram: chỉ flag khi đồng thời đạt count và rate
    ngram_min_doc_count: int = 3
    ngram_min_doc_rate: float = 0.01

    # Candidate review, không phải nhãn ground truth
    relevance_review_threshold: float = 0.15
    long_comment_words: int = 80

    # Split chống leakage
    test_size: float = 0.15
    validation_size_within_train: float = 0.1765  # xấp xỉ 15% toàn bộ

def resolve_default_input() -> str:
    candidates = [
        Path('Preprocessing/data/raw/raw.csv'),
        Path('../data/raw/raw.csv'),
        Path('/kaggle/input/datasets/tungduong2323/raw-comment/raw.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return str(candidates[0])


CFG = AuditConfig(input_file=resolve_default_input())
INPUT_FILE = Path(CFG.input_file)
OUTPUT_DIR = Path(CFG.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CFG.random_seed)
np.random.seed(CFG.random_seed)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
)
log = logging.getLogger('audit_nlp_v2')

print(json.dumps(asdict(CFG), ensure_ascii=False, indent=2))


## 3. Hàm tiện ích, tái lập và xuất file an toàn


In [ ]:
REQUIRED_COLUMNS = ['row_id', 'platform', 'title', 'comment']
ISSUE_COLUMNS = [
    'row_id', 'platform', 'issue_category', 'severity',
    'evidence', 'explanation', 'recommended_action',
]


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:length]


def safe_slug(value: Any, fallback: str = 'unknown') -> str:
    value = str(value).strip().casefold()
    value = unicodedata.normalize('NFD', value)
    value = ''.join(ch for ch in value if unicodedata.category(ch) != 'Mn')
    value = value.replace('đ', 'd')
    value = re.sub(r'[^a-z0-9._-]+', '_', value).strip('._-')
    return value[:80] or fallback


def sanitize_csv_cell(value: Any) -> Any:
    """Trung hòa công thức khi CSV được mở trong Excel/Sheets."""
    if not isinstance(value, str):
        return value
    if CSV_DANGEROUS_PREFIX_RE.match(value):
        return "'" + value
    return value


def safe_to_csv(frame: pd.DataFrame, path: Path) -> None:
    out = frame.copy()
    for col in out.select_dtypes(include=['object', 'string']).columns:
        out[col] = out[col].map(sanitize_csv_cell)
    out.to_csv(path, index=False, encoding='utf-8-sig')


def empty_issues() -> pd.DataFrame:
    return pd.DataFrame(columns=ISSUE_COLUMNS)


def package_versions() -> Dict[str, str]:
    names = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'tqdm', 'regex']
    versions: Dict[str, str] = {
        'python': sys.version.split()[0],
        'platform': py_platform.platform(),
    }
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = 'not-installed'
    return versions


## 4. Chuẩn hóa Unicode, khóa dò rule và token hóa mạng xã hội


In [ ]:
ZERO_WIDTH_RE = uregex.compile(r'[\u200B\u200C\u2060\uFEFF]')
URL_RE = uregex.compile(r'(?i)\b(?:https?://|www\.)\S+')
EMAIL_RE = uregex.compile(r'(?i)\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b')
PHONE_RE = uregex.compile(r'(?<!\d)(?:\+?84|0)(?:[ .-]?\d){9,10}(?!\d)')
MENTION_RE = uregex.compile(r'(?<!\w)@[\p{L}\p{N}_.]+')
FORMULA_LIKE_RE = uregex.compile(r'^\s*(?:=|[+\-@]\s*=)')
CSV_DANGEROUS_PREFIX_RE = uregex.compile(r'^\s*[=+\-@\t\r]')
HASHTAG_RE = uregex.compile(r'(?<!\w)#[\p{L}\p{N}_]+')
WORD_RE = uregex.compile(r"\p{L}+(?:['’]\p{L}+)*|\p{N}+(?:[.,]\p{N}+)*")
GRAPHEME_RE = uregex.compile(r'\X')
EXTENDED_PICTO_RE = uregex.compile(r'\p{Extended_Pictographic}')


def nfc_text(text: Any) -> str:
    text = '' if text is None else str(text)
    text = ZERO_WIDTH_RE.sub(' ', text)
    return unicodedata.normalize('NFC', text)


def normalize_space(text: Any) -> str:
    return uregex.sub(r'\s+', ' ', nfc_text(text)).strip()


def match_key(text: Any) -> str:
    """Khóa dò rule: casefold + bỏ dấu. Không dùng làm đầu vào model."""
    text = normalize_space(text).casefold()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = text.replace('đ', 'd')
    return uregex.sub(r'\s+', ' ', text).strip()


def duplicate_key(text: Any) -> str:
    """Khóa review: NFC, casefold và khoảng trắng; giữ emoji/dấu câu."""
    return normalize_space(text).casefold()


def redact_pii(text: Any) -> str:
    """Xóa PII hiển nhiên khỏi bản xuất dùng để chia sẻ/huấn luyện."""
    clean = normalize_space(text)
    clean = EMAIL_RE.sub(' ', clean)
    clean = PHONE_RE.sub(' ', clean)
    clean = URL_RE.sub(' ', clean)
    clean = MENTION_RE.sub(' ', clean)
    return normalize_space(clean)


def word_tokens(text: Any) -> List[str]:
    return [m.group(0).casefold() for m in WORD_RE.finditer(nfc_text(text))]


def key_tokens(text: Any) -> List[str]:
    return [m.group(0) for m in WORD_RE.finditer(match_key(text))]


def graphemes(text: Any) -> List[str]:
    return GRAPHEME_RE.findall(nfc_text(text))


def extract_emojis(text: Any) -> List[str]:
    return [g for g in graphemes(text) if EXTENDED_PICTO_RE.search(g)]


def phrase_pattern(phrase: str, accent_insensitive: bool = False) -> uregex.Pattern:
    """Ranh giới từ Unicode. Mặc định giữ dấu để không nhập nhằng `cho`/`chó`."""
    normalized = match_key(phrase) if accent_insensitive else normalize_space(phrase).casefold()
    p = uregex.escape(normalized)
    return uregex.compile(rf'(?<![\p{{L}}\p{{N}}_]){p}(?![\p{{L}}\p{{N}}_])', uregex.I)


def find_phrases(
    text: Any, phrases: Iterable[str], accent_insensitive: bool = False
) -> List[str]:
    haystack = match_key(text) if accent_insensitive else normalize_space(text).casefold()
    hits: List[str] = []
    for phrase in phrases:
        if phrase_pattern(phrase, accent_insensitive=accent_insensitive).search(haystack):
            hits.append(phrase)
    return hits


def optional_word_segment(text: Any) -> str:
    """Tách từ tùy chọn; fallback không làm notebook dừng."""
    clean = normalize_space(text)
    if not CFG.enable_word_segmentation:
        return clean
    try:
        from underthesea import word_tokenize
        return word_tokenize(clean, format='text')
    except Exception as exc:
        log.warning('Không dùng được underthesea; fallback văn bản NFC: %s', exc)
        return clean


## 5. Kiểm thử hồi quy lớp tiền xử lý


In [ ]:
def run_preprocessing_tests() -> None:
    assert nfc_text('a\u0301') == 'á'
    assert match_key('Đường phố') == 'duong pho'
    assert find_phrases('Ứng dụng này hữu ích cho sinh viên.', ['chó']) == []
    assert find_phrases('Đồ chó!', ['chó']) == ['chó']
    assert len(extract_emojis('😂😂😂')) == 3
    assert len(extract_emojis('👨‍👩‍👧‍👦')) == 1
    assert sanitize_csv_cell('=2+2') == "'=2+2"
    assert safe_slug('../TikTok Việt') == 'tiktok_viet'

run_preprocessing_tests()
print('✓ Preprocessing regression tests passed')


## 6. Đọc dữ liệu và kiểm tra cấu trúc


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f'Không tìm thấy {INPUT_FILE.resolve()}. Hãy đặt raw.csv cạnh notebook '
        'hoặc đổi CFG.input_file.'
    )

# on_bad_lines='error': không âm thầm bỏ dòng CSV lỗi.
df = pd.read_csv(
    INPUT_FILE,
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    on_bad_lines='error',
)
df.columns = [str(c).strip() for c in df.columns]
missing_columns = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {missing_columns}')

df = df.copy()
df['_source_row_number'] = np.arange(2, len(df) + 2)
for col in REQUIRED_COLUMNS:
    df[col] = df[col].map(nfc_text)

dataset_fingerprint = sha256_file(INPUT_FILE)
log.info('Đã đọc %d dòng | SHA256=%s', len(df), dataset_fingerprint)

structure_issues: List[Dict[str, Any]] = []

def add_structure_issue(row: pd.Series, category: str, severity: str, evidence: str, explanation: str, action: str) -> None:
    structure_issues.append({
        'row_id': row.get('row_id', ''),
        'platform': row.get('platform', ''),
        'issue_category': category,
        'severity': severity,
        'evidence': str(evidence)[:200],
        'explanation': explanation,
        'recommended_action': action,
    })

for _, row in df.iterrows():
    if not row['row_id'].strip():
        add_structure_issue(row, 'missing_row_id', 'critical', row['_source_row_number'],
                            'row_id rỗng', 'Bổ sung row_id duy nhất và ổn định')
    if not row['platform'].strip():
        add_structure_issue(row, 'missing_platform', 'high', row['_source_row_number'],
                            'platform rỗng', 'Bổ sung platform')
    if not row['title'].strip():
        add_structure_issue(row, 'empty_title', 'medium', row['_source_row_number'],
                            'title rỗng', 'Kiểm tra nguồn hoặc quy ước title rỗng')
    if not row['comment'].strip():
        add_structure_issue(row, 'empty_comment', 'critical', row['_source_row_number'],
                            'comment rỗng', 'Loại hoặc bổ sung comment')
    if '\ufffd' in row['title'] or '\ufffd' in row['comment']:
        add_structure_issue(row, 'unicode_replacement_character', 'high', '\ufffd',
                            'Có ký tự thay thế Unicode, có thể do lỗi encoding',
                            'Đọc lại dữ liệu từ nguồn với encoding đúng')
    for col in ['row_id', 'platform', 'title', 'comment']:
        if FORMULA_LIKE_RE.match(str(row[col])):
            add_structure_issue(row, 'csv_formula_injection', 'high', f'{col}=<formula-like prefix>',
                                'Giá trị có thể được spreadsheet diễn giải thành công thức',
                                'Xuất qua safe_to_csv hoặc trung hòa ký tự đầu')

row_id_counts = df['row_id'].value_counts(dropna=False)
for rid, count in row_id_counts[row_id_counts > 1].items():
    sample = df[df['row_id'] == rid].iloc[0]
    add_structure_issue(sample, 'duplicate_row_id', 'critical', f'{rid} × {count}',
                        'row_id không duy nhất', 'Tạo row_id duy nhất trước khi train/split')

structure_issues_df = pd.DataFrame(structure_issues, columns=ISSUE_COLUMNS)
display(structure_issues_df.head(20))
print(f'Structure issues: {len(structure_issues_df)}')


## 7. Tạo các biểu diễn văn bản riêng biệt


In [ ]:
log.info('Đang tạo các biểu diễn văn bản...')
df['title_raw'] = df['title']
df['comment_raw'] = df['comment']
df['title_nfc'] = df['title'].map(normalize_space)
df['comment_nfc'] = df['comment'].map(normalize_space)
df['title_match_key'] = df['title_nfc'].map(match_key)
df['comment_match_key'] = df['comment_nfc'].map(match_key)
df['title_duplicate_key'] = df['title_nfc'].map(duplicate_key)
df['comment_duplicate_key'] = df['comment_nfc'].map(duplicate_key)
df['exact_context_duplicate_key'] = (
    df['platform'].map(normalize_space) + '\x1f'
    + df['title_nfc'] + '\x1f' + df['comment_nfc']
)
df['normalized_context_duplicate_key'] = (
    df['platform'].map(duplicate_key) + '\x1f'
    + df['title_nfc'].map(duplicate_key) + '\x1f'
    + df['comment_duplicate_key']
)
df['title_context_key'] = (
    df['platform'].map(duplicate_key) + '\x1f' + df['title_duplicate_key']
)
df['near_duplicate_block_key'] = df['normalized_context_duplicate_key'].map(
    lambda value: '\x1f'.join(str(value).split('\x1f')[:2])
)
df['title_tokens'] = df['title_nfc'].map(word_tokens)
df['comment_tokens'] = df['comment_nfc'].map(word_tokens)
df['comment_key_tokens'] = df['comment_nfc'].map(key_tokens)
df['comment_model_input'] = df['comment_nfc'].map(optional_word_segment)
df['emojis'] = df['comment_nfc'].map(extract_emojis)
df['emoji_count'] = df['emojis'].map(len)
df['word_count'] = df['comment_tokens'].map(len)
df['char_count'] = df['comment_nfc'].map(len)
df['url_count'] = df['comment_nfc'].map(lambda x: len(URL_RE.findall(x)))
df['mention_count'] = df['comment_nfc'].map(lambda x: len(MENTION_RE.findall(x)))
df['hashtag_count'] = df['comment_nfc'].map(lambda x: len(HASHTAG_RE.findall(x)))

preview_cols = [
    'row_id', 'platform', 'comment_nfc', 'comment_match_key',
    'word_count', 'emoji_count', 'url_count',
]
display(df[preview_cols].head())


## 8. Trùng chính xác và trùng sau chuẩn hóa


In [ ]:
def duplicate_groups(frame: pd.DataFrame, key_col: str, duplicate_type: str) -> pd.DataFrame:
    records: List[Dict[str, Any]] = []
    for key, grp in frame.groupby(key_col, sort=False, dropna=False):
        if not str(key).strip() or len(grp) < 2:
            continue
        records.append({
            'group_id': f'{duplicate_type}_{stable_hash(str(key), 10)}',
            'duplicate_type': duplicate_type,
            'group_size': int(len(grp)),
            'row_ids': json.dumps(grp['row_id'].astype(str).tolist(), ensure_ascii=False),
            'platforms': json.dumps(sorted(grp['platform'].astype(str).unique().tolist()), ensure_ascii=False),
            'distinct_titles': int(grp['title_nfc'].nunique()),
            'example_comment': redact_pii(grp['comment_nfc'].iloc[0][:300]),
            'duplicate_key': str(key)[:500],
        })
    cols = [
        'group_id', 'duplicate_type', 'group_size', 'row_ids', 'platforms',
        'distinct_titles', 'example_comment', 'duplicate_key',
    ]
    return pd.DataFrame(records, columns=cols)

exact_duplicate_groups_df = duplicate_groups(df, 'exact_context_duplicate_key', 'exact_context')
normalized_duplicate_groups_df = duplicate_groups(df, 'normalized_context_duplicate_key', 'normalized_context')

print(f'Exact duplicate groups: {len(exact_duplicate_groups_df)}')
print(f'Normalized duplicate groups: {len(normalized_duplicate_groups_df)}')
display(normalized_duplicate_groups_df.head(20))


## 9. N-gram theo document frequency và tỷ lệ dataset


In [ ]:
STOP_NGRAMS = {
    'cam on ban da chia se',
    'theo y kien cua minh',
    'minh thay cai nay cung',
}


def ngrams(tokens: Sequence[str], n: int) -> List[str]:
    if len(tokens) < n:
        return []
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def detect_ngram_violations(frame: pd.DataFrame) -> pd.DataFrame:
    records: List[Dict[str, Any]] = []
    total = max(len(frame), 1)
    specs = [(5, 'any'), (4, 'opening'), (4, 'ending')]

    for n, position in specs:
        postings: Dict[str, Set[int]] = defaultdict(set)
        for idx, tokens in frame['comment_key_tokens'].items():
            row_ngrams = ngrams(tokens, n)
            if position == 'opening':
                row_ngrams = row_ngrams[:1]
            elif position == 'ending':
                row_ngrams = row_ngrams[-1:]
            for gram in set(row_ngrams):
                if gram not in STOP_NGRAMS:
                    postings[gram].add(idx)

        for gram, indices in postings.items():
            count = len(indices)
            rate = count / total
            if count < CFG.ngram_min_doc_count or rate < CFG.ngram_min_doc_rate:
                continue
            rows = frame.loc[sorted(indices)]
            records.append({
                'ngram': gram,
                'n': n,
                'position': position,
                'document_count': count,
                'document_rate': round(rate, 6),
                'platform_count': int(rows['platform'].nunique()),
                'row_ids': json.dumps(rows['row_id'].astype(str).tolist(), ensure_ascii=False),
            })

    cols = ['ngram', 'n', 'position', 'document_count', 'document_rate', 'platform_count', 'row_ids']
    out = pd.DataFrame(records, columns=cols)
    if len(out):
        out = out.sort_values(['document_rate', 'document_count'], ascending=False).reset_index(drop=True)
    return out

ngram_violations_df = detect_ngram_violations(df)
print(f'N-gram candidates: {len(ngram_violations_df)}')
display(ngram_violations_df.head(30))


## 10. Near-duplicate và template không tạo full ma trận N×N


In [ ]:
_EMBEDDING_MODEL = None


def get_embedding_model():
    global _EMBEDDING_MODEL
    if not CFG.enable_semantic_embeddings:
        return None
    if _EMBEDDING_MODEL is not None:
        return _EMBEDDING_MODEL
    try:
        from sentence_transformers import SentenceTransformer
        _EMBEDDING_MODEL = SentenceTransformer(CFG.embedding_model_name)
        return _EMBEDDING_MODEL
    except Exception as exc:
        log.warning('Semantic embeddings bị tắt do lỗi: %s', exc)
        return None


def _semantic_pair_scores(texts: List[str], pairs: List[Tuple[int, int]]) -> Dict[Tuple[int, int], float]:
    model = get_embedding_model()
    if model is None or not pairs:
        return {}
    emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    return {(i, j): float(np.dot(emb[i], emb[j])) for i, j in pairs}


def near_duplicate_pairs(texts: Sequence[str], block_keys: Optional[Sequence[str]] = None) -> pd.DataFrame:
    columns = [
        'left_index', 'right_index', 'char_similarity', 'semantic_similarity',
        'combined_similarity', 'length_ratio',
    ]
    valid_positions = [i for i, t in enumerate(texts) if len(str(t).strip()) >= CFG.near_dup_min_chars]
    if len(valid_positions) < 2:
        return pd.DataFrame(columns=columns)

    valid_texts = [str(texts[i]) for i in valid_positions]
    try:
        vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=1,
            sublinear_tf=True,
            max_features=50000,
            norm='l2',
        )
        matrix = vectorizer.fit_transform(valid_texts)
    except ValueError as exc:
        if 'empty vocabulary' in str(exc).lower():
            return pd.DataFrame(columns=columns)
        raise

    if matrix.shape[1] == 0:
        return pd.DataFrame(columns=columns)

    k = min(CFG.near_dup_neighbors + 1, len(valid_texts))
    nn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute', n_jobs=CFG.n_jobs)
    nn.fit(matrix)
    distances, neighbors = nn.kneighbors(matrix)

    candidate_rows: List[Dict[str, Any]] = []
    semantic_candidates: List[Tuple[int, int]] = []
    for local_i, (dists, neighs) in enumerate(zip(distances, neighbors)):
        global_i = valid_positions[local_i]
        for dist, local_j in zip(dists[1:], neighs[1:]):
            global_j = valid_positions[int(local_j)]
            if global_j <= global_i:
                continue
            if block_keys is not None and block_keys[global_i] != block_keys[global_j]:
                continue
            char_sim = 1.0 - float(dist)
            if char_sim < CFG.near_dup_char_threshold:
                continue
            a, b = str(texts[global_i]), str(texts[global_j])
            length_ratio = min(len(a), len(b)) / max(len(a), len(b), 1)
            if length_ratio < 0.55:
                continue
            semantic_candidates.append((global_i, global_j))
            candidate_rows.append({
                'left_index': global_i,
                'right_index': global_j,
                'char_similarity': char_sim,
                'semantic_similarity': np.nan,
                'combined_similarity': char_sim,
                'length_ratio': length_ratio,
            })

    semantic = _semantic_pair_scores([str(t) for t in texts], semantic_candidates)
    for row in candidate_rows:
        pair = (row['left_index'], row['right_index'])
        if pair in semantic:
            row['semantic_similarity'] = semantic[pair]
            row['combined_similarity'] = 0.65 * row['char_similarity'] + 0.35 * semantic[pair]

    out = pd.DataFrame(candidate_rows, columns=columns)
    if len(out):
        out = out[out['combined_similarity'] >= CFG.near_dup_combined_threshold]
        out = out.sort_values('combined_similarity', ascending=False).reset_index(drop=True)
    return out


def clique_like_groups(pair_df: pd.DataFrame, n_rows: int) -> List[List[int]]:
    """Greedy complete-link: thành viên mới phải nối với mọi thành viên đã có."""
    adjacency: Dict[int, Set[int]] = defaultdict(set)
    for row in pair_df.itertuples(index=False):
        i, j = int(row.left_index), int(row.right_index)
        adjacency[i].add(j)
        adjacency[j].add(i)

    groups: List[List[int]] = []
    assigned: Set[int] = set()
    seeds = sorted(adjacency, key=lambda x: len(adjacency[x]), reverse=True)
    for seed in seeds:
        if seed in assigned:
            continue
        cluster = [seed]
        candidates = sorted(adjacency[seed], key=lambda x: len(adjacency[x]), reverse=True)
        for candidate in candidates:
            if candidate in assigned:
                continue
            if all(candidate in adjacency[member] for member in cluster):
                cluster.append(candidate)
        if len(cluster) >= 2:
            groups.append(sorted(cluster))
            assigned.update(cluster)
    return groups

near_duplicate_pairs_df = near_duplicate_pairs(
    df['comment_duplicate_key'].tolist(),
    block_keys=df['near_duplicate_block_key'].tolist(),
)
if len(near_duplicate_pairs_df):
    near_duplicate_pairs_df['left_row_id'] = near_duplicate_pairs_df['left_index'].map(df['row_id'])
    near_duplicate_pairs_df['right_row_id'] = near_duplicate_pairs_df['right_index'].map(df['row_id'])
    near_duplicate_pairs_df['left_comment'] = near_duplicate_pairs_df['left_index'].map(df['comment_nfc']).map(redact_pii)
    near_duplicate_pairs_df['right_comment'] = near_duplicate_pairs_df['right_index'].map(df['comment_nfc']).map(redact_pii)

near_groups = clique_like_groups(near_duplicate_pairs_df, len(df))
near_group_records: List[Dict[str, Any]] = []
for seq, indices in enumerate(near_groups, start=1):
    subpairs = near_duplicate_pairs_df[
        near_duplicate_pairs_df['left_index'].isin(indices)
        & near_duplicate_pairs_df['right_index'].isin(indices)
    ]
    near_group_records.append({
        'group_id': f'near_{seq:05d}',
        'group_size': len(indices),
        'row_ids': json.dumps(df.loc[indices, 'row_id'].astype(str).tolist(), ensure_ascii=False),
        'min_pair_similarity': round(float(subpairs['combined_similarity'].min()), 6) if len(subpairs) else np.nan,
        'mean_pair_similarity': round(float(subpairs['combined_similarity'].mean()), 6) if len(subpairs) else np.nan,
        'example_comments': json.dumps([redact_pii(x) for x in df.loc[indices, 'comment_nfc'].head(3)], ensure_ascii=False),
    })
near_duplicate_groups_df = pd.DataFrame(
    near_group_records,
    columns=['group_id', 'group_size', 'row_ids', 'min_pair_similarity', 'mean_pair_similarity', 'example_comments'],
)

print(f'Near-duplicate pairs: {len(near_duplicate_pairs_df)}')
print(f'Clique-like groups: {len(near_duplicate_groups_df)}')
display(near_duplicate_pairs_df.head(20))


## 11. Kiểm thử hồi quy near-duplicate


In [ ]:
def run_near_duplicate_tests() -> None:
    unrelated = [
        'mon nay nhin ngon nhung gia hoi cao',
        'ung dung nay huu ich cho sinh vien',
    ]
    result = near_duplicate_pairs(unrelated)
    assert isinstance(result, pd.DataFrame)
    assert len(result) == 0

    similar = [
        'san pham nay dung kha tot giao hang nhanh',
        'san pham dung kha tot va giao hang nhanh',
    ]
    # Chỉ kiểm tra không crash; threshold có thể thay đổi theo cấu hình.
    result2 = near_duplicate_pairs(similar)
    assert isinstance(result2, pd.DataFrame)

run_near_duplicate_tests()
print('✓ Near-duplicate regression tests passed')


## 12. Title relevance: lexical + character + semantic tùy chọn


In [ ]:
VI_STOPWORDS = {
    'và', 'là', 'của', 'cho', 'có', 'một', 'những', 'các', 'này', 'đó', 'thì',
    'mình', 'tôi', 'bạn', 'ở', 'được', 'với', 'về', 'khi', 'rất', 'cũng', 'đang',
}
VI_STOPWORDS_KEY = {match_key(x) for x in VI_STOPWORDS}


def content_token_set(text: Any) -> Set[str]:
    return {t for t in key_tokens(text) if len(t) > 1 and t not in VI_STOPWORDS_KEY}


def diagonal_char_similarity(left: Sequence[str], right: Sequence[str]) -> np.ndarray:
    n = len(left)
    if n == 0:
        return np.array([], dtype=float)
    corpus = [str(x) for x in left] + [str(x) for x in right]
    try:
        vectorizer = TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3, 5), min_df=1,
            max_features=50000, sublinear_tf=True, norm='l2',
        )
        matrix = vectorizer.fit_transform(corpus)
    except ValueError as exc:
        if 'empty vocabulary' in str(exc).lower():
            return np.zeros(n, dtype=float)
        raise
    a, b = matrix[:n], matrix[n:]
    return np.asarray(a.multiply(b).sum(axis=1)).ravel()


def diagonal_semantic_similarity(left: Sequence[str], right: Sequence[str]) -> Optional[np.ndarray]:
    model = get_embedding_model()
    if model is None or len(left) == 0:
        return None
    left_emb = model.encode(list(left), normalize_embeddings=True, show_progress_bar=True)
    right_emb = model.encode(list(right), normalize_embeddings=True, show_progress_bar=True)
    return np.sum(left_emb * right_emb, axis=1)

lexical_scores: List[float] = []
for title, comment in zip(df['title_nfc'], df['comment_nfc']):
    a, b = content_token_set(title), content_token_set(comment)
    lexical_scores.append(len(a & b) / max(len(a | b), 1) if a else np.nan)

char_scores = diagonal_char_similarity(df['title_nfc'].tolist(), df['comment_nfc'].tolist())
semantic_scores = diagonal_semantic_similarity(df['title_nfc'].tolist(), df['comment_nfc'].tolist())

df['title_relevance_lexical'] = lexical_scores
df['title_relevance_char'] = char_scores
if semantic_scores is None:
    df['title_relevance_semantic'] = np.nan
    df['title_relevance_score'] = (
        0.55 * df['title_relevance_lexical'].fillna(0)
        + 0.45 * df['title_relevance_char'].fillna(0)
    )
else:
    df['title_relevance_semantic'] = semantic_scores
    df['title_relevance_score'] = (
        0.35 * df['title_relevance_lexical'].fillna(0)
        + 0.30 * df['title_relevance_char'].fillna(0)
        + 0.35 * df['title_relevance_semantic'].fillna(0)
    )

df.loc[df['title_nfc'].eq(''), 'title_relevance_score'] = np.nan
df['title_relevance_review'] = (
    df['title_relevance_score'].notna()
    & (df['title_relevance_score'] < CFG.relevance_review_threshold)
)

print('Đây là candidate score; cần gold labels để chọn threshold chính thức.')
display(df[[
    'row_id', 'title_nfc', 'comment_nfc', 'title_relevance_lexical',
    'title_relevance_char', 'title_relevance_semantic',
    'title_relevance_score', 'title_relevance_review',
]].sort_values('title_relevance_score').head(20))


## 13. Rule signals cho offensive language, targeted attack và safety


In [ ]:
OFFENSIVE_TERMS: Dict[str, List[str]] = {
    'profanity_strong': [
        'địt mẹ', 'đụ má', 'đcm', 'dcm', 'dkm', 'vcl', 'clm', 'cái lồn', 'cái cặc',
    ],
    'insult_explicit': [
        'óc chó', 'đồ chó', 'ngu xuẩn', 'mất dạy', 'vô học', 'thiểu năng',
        'rác rưởi', 'vô dụng', 'cặn bã', 'khốn nạn',
    ],
    'insult_contextual': [
        'ngu', 'đần', 'dốt', 'vô duyên', 'bất lịch sự', 'hèn hạ',
    ],
}

TARGET_MARKERS = [
    'mày', 'may', 'mi', 'nó', 'no', 'thằng đó', 'thang do', 'con đó', 'con do', 'bọn mày', 'bon may', 'tụi mày', 'tui may',
]

THREAT_PATTERNS = [
    uregex.compile(r'(?<!\w)(tao|toi|tui|minh)\s+(se\s+)?(giet|dam|chem|danh)\s+(may|m|no|nguoi do)(?!\w)', uregex.I),
    uregex.compile(r'(?<!\w)(giet|danh|dap|chem)\s+(no|may|thang do|con do)\s*(di|cho tao)?(?!\w)', uregex.I),
    uregex.compile(r'(?<!\w)cho\s+(may|no)\s+an\s+don(?!\w)', uregex.I),
]

ILLEGAL_GUIDE_PHRASES = [
    'cách làm bom', 'cách chế vũ khí', 'cách hack tài khoản',
    'mua ma túy ở đâu', 'mua vũ khí ở đâu',
]

HARASSMENT_CALL_PHRASES = [
    'mass report', 'kêu gọi report', 'mọi người vào chửi', 'tấn công tài khoản này',
]

ID_RE = uregex.compile(r'(?<!\d)(?:\d{9}|\d{12})(?!\d)')
PII_CONTEXT_PHRASES = ['số điện thoại', 'địa chỉ nhà', 'cccd', 'căn cước', 'tài khoản ngân hàng']
DOXXING_ACTION_PHRASES = ['đăng lên', 'phát tán', 'chia sẻ số', 'công khai địa chỉ', 'bóc địa chỉ']


def analyze_rule_signals(title: Any, comment: Any) -> Dict[str, Any]:
    combined = normalize_space(f'{title} {comment}')
    key = match_key(combined)
    hits: Dict[str, List[str]] = {}
    for category, terms in OFFENSIVE_TERMS.items():
        matched = find_phrases(combined, terms)
        if matched:
            hits[category] = matched

    target_hits = find_phrases(combined, TARGET_MARKERS) + MENTION_RE.findall(combined)
    targeted_attack = bool(target_hits) and bool(hits.get('insult_explicit') or hits.get('insult_contextual'))

    threat_evidence = []
    for pat in THREAT_PATTERNS:
        match = pat.search(key)
        if match:
            threat_evidence.append(match.group(0))

    illegal_hits = find_phrases(combined, ILLEGAL_GUIDE_PHRASES)
    harassment_hits = find_phrases(combined, HARASSMENT_CALL_PHRASES)
    pii_hits: List[str] = []
    pii_hits.extend(PHONE_RE.findall(combined))
    pii_hits.extend(EMAIL_RE.findall(combined))
    pii_hits.extend(find_phrases(combined, PII_CONTEXT_PHRASES))
    doxxing = bool(pii_hits) and bool(find_phrases(combined, DOXXING_ACTION_PHRASES))

    if threat_evidence or illegal_hits or doxxing:
        risk = 'high'
    elif targeted_attack or hits.get('profanity_strong') or harassment_hits:
        risk = 'medium'
    elif hits:
        risk = 'low'
    else:
        risk = 'none'

    return {
        'rule_risk': risk,
        'offensive_hits': hits,
        'target_markers': target_hits,
        'targeted_attack_candidate': targeted_attack,
        'threat_evidence': threat_evidence,
        'illegal_guide_hits': illegal_hits,
        'harassment_call_hits': harassment_hits,
        'pii_hits': pii_hits,
        'doxxing_candidate': doxxing,
    }

rule_results = [analyze_rule_signals(t, c) for t, c in zip(df['title_nfc'], df['comment_nfc'])]
df['rule_risk'] = [r['rule_risk'] for r in rule_results]
df['offensive_hits'] = [json.dumps(r['offensive_hits'], ensure_ascii=False) for r in rule_results]
df['targeted_attack_candidate'] = [r['targeted_attack_candidate'] for r in rule_results]
df['threat_evidence'] = [json.dumps(r['threat_evidence'], ensure_ascii=False) for r in rule_results]
df['illegal_guide_hits'] = [json.dumps(r['illegal_guide_hits'], ensure_ascii=False) for r in rule_results]
df['harassment_call_hits'] = [json.dumps(r['harassment_call_hits'], ensure_ascii=False) for r in rule_results]
df['pii_hits'] = [json.dumps(r['pii_hits'], ensure_ascii=False) for r in rule_results]
df['doxxing_candidate'] = [r['doxxing_candidate'] for r in rule_results]

display(df[['row_id', 'comment_nfc', 'rule_risk', 'offensive_hits',
            'targeted_attack_candidate', 'threat_evidence', 'pii_hits']]
        .query("rule_risk != 'none'").head(30))


## 14. Kiểm thử hồi quy toxicity/safety


In [ ]:
def run_rule_tests() -> None:
    clean = analyze_rule_signals('', 'Ứng dụng này hữu ích cho sinh viên.')
    assert clean['rule_risk'] == 'none'
    assert not clean['targeted_attack_candidate']

    insult = analyze_rule_signals('', 'Mày đúng là đồ chó.')
    assert insult['targeted_attack_candidate']
    assert insult['rule_risk'] in {'medium', 'high'}

    neutral_violence = analyze_rule_signals('', 'Phim có cảnh đánh nhau khá nhiều.')
    assert neutral_violence['threat_evidence'] == []

    threat = analyze_rule_signals('', 'Tao sẽ đánh mày.')
    assert threat['threat_evidence']

run_rule_tests()
print('✓ Toxicity/safety regression tests passed')


## 15. Teencode và slang: đếm theo token, không tự động thay thế


In [ ]:
TEENCODE_CONFIDENT = {
    'ko', 'khum', 'hong', 'hok', 'dc', 'đc', 'ntn', 'mn', 'bth', 'tks', 'thx', 'rep',
}
TEENCODE_AMBIGUOUS = {
    'k', 'm', 'n', 't', 'v', 'z', 'j', 'ad', 'bt', 'vs',
}
SLANG_TERMS = {
    'slay', 'chill', 'vibe', 'flex', 'lowkey', 'highkey', 'simp', 'rizz', 'goat',
    'cringe', 'fomo', 'viral', 'trend', 'bóc phốt', 'cực phẩm', 'thả tim',
}
TEENCODE_CONFIDENT_KEY = {match_key(x) for x in TEENCODE_CONFIDENT}
TEENCODE_AMBIGUOUS_KEY = {match_key(x) for x in TEENCODE_AMBIGUOUS}


def analyze_teencode_slang(text: Any) -> Dict[str, Any]:
    tokens = key_tokens(text)
    confident = [t for t in tokens if t in TEENCODE_CONFIDENT_KEY]
    ambiguous = [t for t in tokens if t in TEENCODE_AMBIGUOUS_KEY]
    slang = find_phrases(text, SLANG_TERMS)
    denom = max(len(tokens), 1)
    return {
        'teencode_confident_tokens': confident,
        'teencode_ambiguous_tokens': ambiguous,
        'slang_terms': slang,
        'teencode_token_rate': len(confident) / denom,
        'slang_token_rate': len(slang) / denom,
    }

tc_results = [analyze_teencode_slang(c) for c in df['comment_nfc']]
df['teencode_confident_tokens'] = [json.dumps(x['teencode_confident_tokens'], ensure_ascii=False) for x in tc_results]
df['teencode_ambiguous_tokens'] = [json.dumps(x['teencode_ambiguous_tokens'], ensure_ascii=False) for x in tc_results]
df['slang_terms'] = [json.dumps(x['slang_terms'], ensure_ascii=False) for x in tc_results]
df['teencode_token_rate'] = [x['teencode_token_rate'] for x in tc_results]
df['slang_token_rate'] = [x['slang_token_rate'] for x in tc_results]
df['has_teencode_confident'] = df['teencode_token_rate'] > 0
df['has_slang'] = df['slang_token_rate'] > 0

print(f"Row rate có teencode chắc chắn: {df['has_teencode_confident'].mean():.2%}")
print(f"Token rate teencode toàn tập: {sum(len(json.loads(x)) for x in df['teencode_confident_tokens']) / max(df['word_count'].sum(), 1):.2%}")
print(f"Row rate có slang: {df['has_slang'].mean():.2%}")


## 16. Chất lượng định dạng — không gọi là “naturalness”


In [ ]:
REPEATED_CHAR_RE = uregex.compile(r'(.)\1{4,}', uregex.S)
EXCESSIVE_PUNCT_RE = uregex.compile(r'([!?.,])\1{2,}')
LAUGHTER_ONLY_RE = uregex.compile(r'(?i)^\s*(?:ha|hi|he|kk|kaka|haha|hihi|hehe)[.! ]*\s*$')


def format_quality(text: Any, word_count_value: int, emoji_count_value: int) -> Tuple[int, List[str]]:
    text = normalize_space(text)
    flags: List[str] = []
    score = 100
    if not text:
        return 0, ['empty_comment']
    if word_count_value == 0 and emoji_count_value > 0:
        flags.append('emoji_only'); score -= 25
    if word_count_value < 2:
        flags.append('very_short'); score -= 15
    if word_count_value > CFG.long_comment_words:
        flags.append('very_long'); score -= 10
    if EXCESSIVE_PUNCT_RE.search(text):
        flags.append('excessive_punctuation'); score -= 10
    if REPEATED_CHAR_RE.search(text):
        flags.append('repeated_characters'); score -= 10
    if LAUGHTER_ONLY_RE.match(text):
        flags.append('laughter_only'); score -= 15
    if emoji_count_value >= 6:
        flags.append('many_emojis'); score -= 10
    return max(score, 0), flags

format_results = [
    format_quality(c, int(w), int(e))
    for c, w, e in zip(df['comment_nfc'], df['word_count'], df['emoji_count'])
]
df['format_quality_score'] = [x[0] for x in format_results]
df['format_flags'] = [json.dumps(x[1], ensure_ascii=False) for x in format_results]

platform_metrics_df = df.groupby('platform', dropna=False).agg(
    row_count=('row_id', 'size'),
    mean_words=('word_count', 'mean'),
    median_words=('word_count', 'median'),
    p10_words=('word_count', lambda x: x.quantile(0.10)),
    p90_words=('word_count', lambda x: x.quantile(0.90)),
    emoji_row_rate=('emoji_count', lambda x: (x > 0).mean()),
    teencode_row_rate=('has_teencode_confident', 'mean'),
    slang_row_rate=('has_slang', 'mean'),
    rule_review_rate=('rule_risk', lambda x: x.ne('none').mean()),
    low_relevance_candidate_rate=('title_relevance_review', 'mean'),
).reset_index()

for col in platform_metrics_df.select_dtypes(include='number').columns:
    platform_metrics_df[col] = platform_metrics_df[col].round(6)

display(platform_metrics_df)


## 17. Gom quality issues theo từng detector


In [ ]:
quality_issue_records: List[Dict[str, Any]] = []
if len(structure_issues_df):
    quality_issue_records.extend(structure_issues_df.to_dict('records'))


def add_issue(row: pd.Series, category: str, severity: str, evidence: Any, explanation: str, action: str) -> None:
    quality_issue_records.append({
        'row_id': row['row_id'],
        'platform': row['platform'],
        'issue_category': category,
        'severity': severity,
        'evidence': str(evidence)[:300],
        'explanation': explanation,
        'recommended_action': action,
    })

# Row-level duplicate mapping
for table, category, severity in [
    (exact_duplicate_groups_df, 'exact_duplicate', 'high'),
    (normalized_duplicate_groups_df, 'normalized_duplicate', 'medium'),
    (near_duplicate_groups_df, 'near_duplicate_template', 'medium'),
]:
    for rec in table.to_dict('records'):
        for rid in json.loads(rec['row_ids']):
            matches = df[df['row_id'].astype(str) == str(rid)]
            if matches.empty:
                continue
            row = matches.iloc[0]
            add_issue(row, category, severity, rec['group_id'],
                      f"Thuộc nhóm {rec['group_id']} kích thước {rec['group_size']}",
                      'Giữ cùng split; kiểm tra loại bỏ hoặc đa dạng hóa')

for _, row in df[df['title_relevance_review']].iterrows():
    add_issue(row, 'low_title_relevance_candidate', 'medium',
              round(float(row['title_relevance_score']), 4),
              'Điểm relevance heuristic dưới threshold cấu hình',
              'Đưa vào mẫu review; không tự động loại nếu chưa có gold label')

for _, row in df[df['rule_risk'].ne('none')].iterrows():
    severity = {'low': 'low', 'medium': 'high', 'high': 'critical'}[row['rule_risk']]
    add_issue(row, 'language_or_safety_rule_signal', severity,
              row['offensive_hits'] + ' ' + row['threat_evidence'],
              f"Rule risk={row['rule_risk']}; đây là candidate cho human review",
              'Review theo guideline; không coi rule là ground truth')

for _, row in df[df['format_quality_score'] < 70].iterrows():
    add_issue(row, 'format_quality', 'low', row['format_flags'],
              f"Format quality score={row['format_quality_score']}",
              'Kiểm tra định dạng; không suy diễn thành độ tự nhiên ngôn ngữ')

# N-gram issue theo từng row, nhưng giữ một issue cho mỗi n-gram.
for rec in ngram_violations_df.to_dict('records'):
    for rid in json.loads(rec['row_ids']):
        matches = df[df['row_id'].astype(str) == str(rid)]
        if matches.empty:
            continue
        row = matches.iloc[0]
        add_issue(row, 'repeated_ngram_candidate', 'low', rec['ngram'],
                  f"{rec['position']} {rec['n']}-gram xuất hiện ở {rec['document_rate']:.2%} số dòng",
                  'Kiểm tra boilerplate/template theo ngữ cảnh platform')

quality_issues_df = pd.DataFrame(quality_issue_records, columns=ISSUE_COLUMNS)
if len(quality_issues_df):
    quality_issues_df = quality_issues_df.drop_duplicates(
        subset=['row_id', 'issue_category', 'evidence']
    ).reset_index(drop=True)

print(f'Quality issues: {len(quality_issues_df)}')
display(quality_issues_df.head(30))


## 18. Đánh giá bằng gold labels và hiệu chuẩn threshold — nếu có


In [ ]:
def evaluate_binary(y_true: Sequence[int], y_pred: Sequence[int], name: str) -> Dict[str, Any]:
    y_true_arr = np.asarray(y_true, dtype=int)
    y_pred_arr = np.asarray(y_pred, dtype=int)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_arr, y_pred_arr, labels=[0, 1], zero_division=0
    )
    result = {
        'name': name,
        'confusion_matrix': confusion_matrix(y_true_arr, y_pred_arr, labels=[0, 1]).tolist(),
        'precision_class_0': float(precision[0]),
        'recall_class_0': float(recall[0]),
        'f1_class_0': float(f1[0]),
        'support_class_0': int(support[0]),
        'precision_class_1': float(precision[1]),
        'recall_class_1': float(recall[1]),
        'f1_class_1': float(f1[1]),
        'support_class_1': int(support[1]),
        'macro_f1': float(f1_score(y_true_arr, y_pred_arr, average='macro', zero_division=0)),
        'cohen_kappa': float(cohen_kappa_score(y_true_arr, y_pred_arr)),
    }
    return result


def best_relevance_threshold(y_true: Sequence[int], scores: Sequence[float]) -> Tuple[float, float]:
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    best_t, best_f1 = 0.0, -1.0
    for threshold in np.linspace(0.0, 1.0, 201):
        pred = (s >= threshold).astype(int)
        score = f1_score(y, pred, average='macro', zero_division=0)
        if score > best_f1:
            best_t, best_f1 = float(threshold), float(score)
    return best_t, best_f1

GOLD_FILE = Path(CFG.gold_labels_file)
evaluation_metrics: Dict[str, Any] = {}
if GOLD_FILE.exists():
    gold = pd.read_csv(GOLD_FILE, dtype=str, encoding='utf-8-sig', keep_default_na=False)
    if 'row_id' not in gold.columns:
        raise ValueError('gold_labels.csv phải có cột row_id')
    if gold['row_id'].duplicated().any():
        raise ValueError('gold_labels.csv có row_id trùng; cần một nhãn duy nhất cho mỗi dòng')
    eval_df = df.merge(gold, on='row_id', how='inner', suffixes=('', '_gold'), validate='one_to_one')
    print(f'Gold matched rows: {len(eval_df)}')

    def validated_binary(values: pd.Series, column: str) -> np.ndarray:
        numeric = pd.to_numeric(values, errors='raise').astype(int)
        invalid = set(numeric.unique()) - {0, 1}
        if invalid:
            raise ValueError(f'{column} chỉ được chứa 0/1; gặp {sorted(invalid)}')
        return numeric.to_numpy()

    if 'toxicity_gold' in eval_df.columns:
        y_true = validated_binary(eval_df['toxicity_gold'], 'toxicity_gold')
        y_pred = eval_df['rule_risk'].isin(['medium', 'high']).astype(int).to_numpy()
        evaluation_metrics['toxicity_rule'] = evaluate_binary(y_true, y_pred, 'toxicity_rule')

    if 'relevance_gold' in eval_df.columns:
        valid = eval_df['title_relevance_score'].notna()
        relevant = eval_df.loc[valid].reset_index(drop=True)
        if len(relevant) >= 10:
            rng = np.random.default_rng(CFG.random_seed)
            order = rng.permutation(len(relevant))
            calibration_size = max(1, min(len(relevant) - 1, len(relevant) // 2))
            calibration = relevant.iloc[order[:calibration_size]]
            evaluation = relevant.iloc[order[calibration_size:]]
            y_cal = validated_binary(calibration['relevance_gold'], 'relevance_gold')
            threshold, calibration_macro_f1 = best_relevance_threshold(
                y_cal, calibration['title_relevance_score'].astype(float)
            )
            y_true = validated_binary(evaluation['relevance_gold'], 'relevance_gold')
            y_pred = (evaluation['title_relevance_score'].astype(float) >= threshold).astype(int)
            evaluation_metrics['relevance'] = evaluate_binary(y_true, y_pred, 'relevance')
            evaluation_metrics['relevance']['calibrated_threshold'] = threshold
            evaluation_metrics['relevance']['calibration_macro_f1'] = calibration_macro_f1
            evaluation_metrics['relevance']['calibration_rows'] = len(calibration)
            evaluation_metrics['relevance']['evaluation_rows'] = len(evaluation)
        else:
            print('Gold relevance có dưới 10 dòng hợp lệ; bỏ qua hiệu chuẩn.')

    display(pd.json_normalize(evaluation_metrics).T)
else:
    print(
        'Không có gold_labels.csv. Các cờ hiện tại chỉ là candidate review. '
        'Nên tạo gold set có row_id và các cột toxicity_gold/relevance_gold.'
    )


## 19. Tạo group chống leakage và chia train/validation/test


In [ ]:
class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

uf = UnionFind(len(df))
row_id_to_indices: Dict[str, List[int]] = defaultdict(list)
for idx, rid in enumerate(df['row_id'].astype(str)):
    row_id_to_indices[rid].append(idx)

# Exact/normalized duplicate groups
for table in [exact_duplicate_groups_df, normalized_duplicate_groups_df]:
    for rec in table.to_dict('records'):
        indices: List[int] = []
        for rid in json.loads(rec['row_ids']):
            indices.extend(row_id_to_indices.get(str(rid), []))
        for i in indices[1:]:
            uf.union(indices[0], i)

# Near-duplicate pairs: chaining được chấp nhận ở đây vì mục tiêu là chống leakage.
for rec in near_duplicate_pairs_df.to_dict('records'):
    uf.union(int(rec['left_index']), int(rec['right_index']))

# Các comment cùng title cũng không nên rơi sang nhiều split nếu title đại diện cùng post.
for _, indices in df.groupby('title_context_key').groups.items():
    indices = list(indices)
    if len(indices) > 1 and df.loc[indices[0], 'title_duplicate_key']:
        for i in indices[1:]:
            uf.union(indices[0], i)

df['leakage_group_id'] = [f'g_{stable_hash(str(uf.find(i)), 12)}' for i in range(len(df))]

split_assignments = pd.DataFrame({
    'row_id': df['row_id'],
    'leakage_group_id': df['leakage_group_id'],
    'split': 'unassigned',
})

unique_groups = df['leakage_group_id'].nunique()
if len(df) >= 10 and unique_groups >= 3:
    indices = np.arange(len(df))
    first = GroupShuffleSplit(
        n_splits=1, test_size=CFG.test_size, random_state=CFG.random_seed
    )
    train_val_idx, test_idx = next(first.split(indices, groups=df['leakage_group_id']))

    train_val_groups = df.iloc[train_val_idx]['leakage_group_id']
    if train_val_groups.nunique() >= 2:
        second = GroupShuffleSplit(
            n_splits=1,
            test_size=CFG.validation_size_within_train,
            random_state=CFG.random_seed + 1,
        )
        rel_train, rel_val = next(second.split(train_val_idx, groups=train_val_groups))
        train_idx = train_val_idx[rel_train]
        val_idx = train_val_idx[rel_val]
    else:
        train_idx, val_idx = train_val_idx, np.array([], dtype=int)

    split_assignments.loc[train_idx, 'split'] = 'train'
    split_assignments.loc[val_idx, 'split'] = 'validation'
    split_assignments.loc[test_idx, 'split'] = 'test'
else:
    # Dataset nhỏ: gán group bằng hash để vẫn bảo toàn group.
    group_split = {}
    choices = ['train', 'validation', 'test']
    weights = [0.70, 0.15, 0.15]
    for group in sorted(df['leakage_group_id'].unique()):
        value = int(stable_hash(group, 8), 16) / 0xFFFFFFFF
        group_split[group] = 'train' if value < weights[0] else ('validation' if value < sum(weights[:2]) else 'test')
    split_assignments['split'] = split_assignments['leakage_group_id'].map(group_split)

leak_check = split_assignments.groupby('leakage_group_id')['split'].nunique().max()
assert leak_check == 1, 'Leakage group bị phân tán qua nhiều split'

display(split_assignments['split'].value_counts().to_frame())

split_counts = split_assignments['split'].value_counts().reindex(
    ['train', 'validation', 'test'], fill_value=0
)
split_rates = (split_counts / max(len(split_assignments), 1)).to_dict()
target_rates = {'train': 1 - CFG.test_size - 0.15, 'validation': 0.15, 'test': CFG.test_size}
split_deviation = {
    key: float(split_rates.get(key, 0.0) - target_rates[key])
    for key in target_rates
}
if max(abs(value) for value in split_deviation.values()) > 0.05:
    log.warning('Tỷ lệ split lệch mục tiêu do group lớn: %s', split_rates)



## 20. Trạng thái audit minh bạch, không giả làm metric model


In [ ]:
critical_count = int((quality_issues_df['severity'] == 'critical').sum()) if len(quality_issues_df) else 0
high_count = int((quality_issues_df['severity'] == 'high').sum()) if len(quality_issues_df) else 0
manual_review_rows = int(quality_issues_df['row_id'].astype(str).nunique()) if len(quality_issues_df) else 0

if len(df) == 0 or critical_count > 0:
    audit_status = 'BLOCKED'
elif not evaluation_metrics:
    audit_status = 'VALIDATION_REQUIRED'
elif high_count > 0:
    audit_status = 'REVIEW_REQUIRED'
else:
    audit_status = 'READY_FOR_TRAINING_REVIEW'

summary_metrics: Dict[str, Any] = {
    'audit_version': '2.0',
    'status': audit_status,
    'dataset_fingerprint_sha256': dataset_fingerprint,
    'total_rows': int(len(df)),
    'valid_nonempty_comment_rows': int(df['comment_nfc'].ne('').sum()),
    'platform_counts': {str(k): int(v) for k, v in df['platform'].value_counts().items()},
    'missing_row_id_count': int(df['row_id'].eq('').sum()),
    'duplicate_row_id_count': int(df['row_id'].duplicated(keep=False).sum()),
    'exact_duplicate_group_count': int(len(exact_duplicate_groups_df)),
    'normalized_duplicate_group_count': int(len(normalized_duplicate_groups_df)),
    'near_duplicate_pair_count': int(len(near_duplicate_pairs_df)),
    'near_duplicate_group_count': int(len(near_duplicate_groups_df)),
    'ngram_candidate_count': int(len(ngram_violations_df)),
    'low_relevance_candidate_rate': float(df['title_relevance_review'].mean()),
    'rule_signal_row_rate': float(df['rule_risk'].ne('none').mean()),
    'teencode_confident_row_rate': float(df['has_teencode_confident'].mean()),
    'teencode_confident_token_rate': float(
        sum(len(json.loads(x)) for x in df['teencode_confident_tokens']) / max(df['word_count'].sum(), 1)
    ),
    'slang_row_rate': float(df['has_slang'].mean()),
    'emoji_row_rate': float((df['emoji_count'] > 0).mean()),
    'critical_issue_count': critical_count,
    'high_issue_count': high_count,
    'manual_review_row_count': manual_review_rows,
    'evaluation_available': bool(evaluation_metrics),
    'evaluation_metrics': evaluation_metrics,
    'split_counts': {str(k): int(v) for k, v in split_counts.items()},
    'split_rates': {str(k): float(v) for k, v in split_rates.items()},
    'split_target_rates': target_rates,
    'split_deviation': split_deviation,
}

print(json.dumps(summary_metrics, ensure_ascii=False, indent=2))


## 21. Xuất kết quả, config, phiên bản và báo cáo


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
created_files: List[Path] = []

public_columns = [
    'row_id', 'platform', 'title_clean', 'comment_clean',
    'emoji_count', 'word_count', 'char_count', 'url_count', 'mention_count', 'hashtag_count',
    'title_relevance_score', 'title_relevance_review', 'rule_risk',
    'targeted_attack_candidate', 'doxxing_candidate', 'has_teencode_confident', 'has_slang',
    'format_quality_score', 'format_flags', 'leakage_group_id',
]
public_df = df.copy()
public_df['title_clean'] = df['title'].map(redact_pii)
public_df['comment_clean'] = df['comment'].map(redact_pii)
public_df = public_df[[column for column in public_columns if column in public_df.columns]]

exports = {
    'enriched_rows.csv': public_df,
    'quality_issues.csv': quality_issues_df,
    'exact_duplicate_groups.csv': exact_duplicate_groups_df,
    'normalized_duplicate_groups.csv': normalized_duplicate_groups_df,
    'near_duplicate_pairs.csv': near_duplicate_pairs_df,
    'near_duplicate_groups.csv': near_duplicate_groups_df,
    'ngram_violations.csv': ngram_violations_df,
    'platform_metrics.csv': platform_metrics_df,
    'split_assignments.csv': split_assignments,
}

def redact_export(frame: pd.DataFrame) -> pd.DataFrame:
    '''Redact PII ở cột nội dung, giữ nguyên ID và mapping split.'''
    out = frame.copy()
    identifier_columns = {
        'row_id', 'platform', 'group_id', 'leakage_group_id',
        'row_ids', 'left_row_id', 'right_row_id', 'duplicate_type', 'split',
    }
    for column in out.select_dtypes(include=['object', 'string']).columns:
        if column not in identifier_columns:
            out[column] = out[column].map(redact_pii)
    return out

for filename, frame in exports.items():
    path = OUTPUT_DIR / filename
    safe_to_csv(redact_export(frame), path)
    created_files.append(path)
json_outputs = {
    'quality_metrics.json': summary_metrics,
    'config.json': asdict(CFG),
    'library_versions.json': package_versions(),
}
for filename, payload in json_outputs.items():
    path = OUTPUT_DIR / filename
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    created_files.append(path)

# Dashboard đơn giản, tên file platform được slug hóa.
fig, ax = plt.subplots(figsize=(9, 5))
platform_plot = platform_metrics_df.sort_values('row_count', ascending=False)
ax.bar(platform_plot['platform'].astype(str), platform_plot['row_count'])
ax.set_title('Số dòng theo platform')
ax.set_ylabel('Số dòng')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plot_path = OUTPUT_DIR / 'platform_row_counts.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)
created_files.append(plot_path)

report_lines = [
    '# Báo cáo Audit Social Comments NLP v2',
    '',
    f'- **Input:** `{INPUT_FILE}`',
    f'- **SHA256:** `{dataset_fingerprint}`',
    f'- **Rows:** {len(df):,}',
    f'- **Status:** **{audit_status}**',
    f'- **Gold evaluation available:** {bool(evaluation_metrics)}',
    '',
    '## Cách diễn giải',
    '',
    '- `rule_signal`, `low_relevance_candidate` và `near_duplicate` là tín hiệu review, không phải ground truth.',
    '- Chỉ dùng threshold chính thức sau khi hiệu chuẩn trên gold validation set.',
    '- Các dòng trong cùng `leakage_group_id` luôn ở cùng một split.',
    '',
    '## Chỉ số chính',
    '',
]
for key, value in summary_metrics.items():
    if key != 'evaluation_metrics':
        report_lines.append(f'- **{key}:** {value}')
report_lines.extend([
    '',
    '## Việc cần làm tiếp',
    '',
    '1. Review mẫu phân tầng từ `quality_issues.csv`.',
    '2. Tạo `gold_labels.csv` và chạy lại để đo precision/recall/F1/kappa.',
    '3. Chỉ chốt ngưỡng và trạng thái training sau bước hiệu chuẩn.',
])
report_path = OUTPUT_DIR / 'quality_audit_report.md'
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
created_files.append(report_path)

print('Files created:')
for path in created_files:
    print(' ✓', path.resolve())


## 22. Kết luận

Notebook này cố ý phân biệt ba khái niệm:

1. **Tín hiệu rule**: dùng để ưu tiên human review.
2. **Metric detector/model**: chỉ có sau khi so với gold labels.
3. **Trạng thái dataset**: quyết định vận hành, không phải độ chính xác NLP.

Không có cell tự chạy script bên ngoài. Chạy `Run All` chỉ đọc file cấu hình, phân tích dữ liệu và ghi kết quả vào `audit_output_v2/`.
